# Design Notes — RL Agent for PettingZoo Atari *Warlords*

This notebook is a **template** for documenting the design decisions behind the implementation in this repo. Each section follows the same structure:

1. **Problem** — what question are we answering?
2. **Options considered** — the realistic alternatives.
3. **Decision** — what was chosen.
4. **Why** — the reasoning (to be backed by sources you'll add).
5. **Where it lives in the code** — file / function.
6. **Sources / further reading** — placeholders for you to fill in.

The flow follows the actual *path of discovery* in this project:

> Pick an RL algorithm → realize PPO has known numerical-stability issues with reward scale → discover HEPPO and adopt its standardization/quantization tricks → notice that *Warlords*' reward is delayed terminal (+1/-1 once per episode), forcing changes to credit assignment → realize raw return is no longer a valid skill metric under self-play → adopt ELO + an opponent pool.

## 0. Environment & problem setup

**Environment:** `pettingzoo.atari.warlords_v3` with RAM observations.

- 4 players, symmetric roles, each defends a fortress.
- Observation: 128-byte RAM vector per agent.
- Action space: 6 discrete actions.
- **Reward structure (critical for downstream decisions):** 0 every step, then a single terminal **+1 (win) / -1 (lose)** when a fortress falls or the agent is last standing. Episodes are long (hundreds of steps).
- Multi-agent, competitive, zero-sum-ish.

Every decision below is downstream of these facts — write them down first so the rest reads as consequences, not arbitrary choices.

*Sources to add:*
- [ ] PettingZoo Atari docs for `warlords_v3` (observation / reward spec).
- [ ] Original Atari Warlords game description (for the win/lose semantics).


In [ ]:
# Quick layout check — what's in the repo?
import os
for root, dirs, files in os.walk('.', topdown=True):
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
    depth = root.count(os.sep)
    print('  ' * depth + os.path.basename(root) + '/')
    for f in sorted(files):
        if f.endswith(('.py', '.md')):
            print('  ' * (depth + 1) + f)

## 1. Choosing the RL algorithm: why PPO?

**Problem.** We need a deep-RL algorithm that can train a stochastic policy on a discrete-action, partially-observable, multi-agent Atari environment.

**Options considered.**

| Family | Example | Pros | Cons in our setting |
|---|---|---|---|
| Value-based | **DQN**, Double-DQN, Rainbow | sample efficient, well-studied on Atari | off-policy + replay buffer is awkward under self-play (stale opponents in buffer); ε-greedy is a poor fit for adversarial play; no native stochastic policy |
| On-policy actor-critic | **A2C / A3C** | simple, stable on-policy | no trust-region constraint → larger update steps can collapse the policy under noisy, sparse rewards |
| Trust-region policy gradient | **TRPO** | strong theoretical guarantees | second-order; expensive; harder to implement cleanly with parameter sharing |
| Clipped policy gradient | **PPO** | first-order, simple, stochastic policy, clipped trust region, strong empirical track record on Atari & multi-agent | hyperparameter sensitive; needs care with reward scaling |
| Off-policy actor-critic (continuous) | SAC, DDPG, TD3 | sample efficient | designed for continuous control; not the natural fit for discrete Atari actions |
| Model-based / planning | MuZero, Dreamer | very strong on Atari | much higher implementation complexity; overkill for a course-scale project |

**Decision.** Use **PPO-Clip**.

**Why (to source).**
- PPO is the de-facto baseline for on-policy deep RL — simple, robust, and a standard choice for multi-agent self-play setups (it's what AlphaStar / OpenAI Five build on).
- Naturally outputs a *stochastic* policy, which is what self-play needs (deterministic policies in self-play tend to cycle / get exploited).
- On-policy → the rollout always reflects the *current* opponent matchup, so there is no replay-buffer staleness problem when opponents change.
- First-order, so it composes cleanly with parameter sharing across the 4 symmetric agent slots.

**Where it lives.** `ppo_agent.py` (the PPO-Clip update is in `HEPPOAgent.update`).

*Sources to add:*
- [ ] Schulman et al. 2017, *Proximal Policy Optimization Algorithms* (arXiv:1707.06347).
- [ ] Mnih et al. 2013/2015, *Playing Atari with Deep Reinforcement Learning* / Nature DQN.
- [ ] Mnih et al. 2016, *Asynchronous Methods for Deep RL* (A2C/A3C).
- [ ] Schulman et al. 2015, *Trust Region Policy Optimization*.
- [ ] Engstrom et al. 2020, *Implementation Matters in Deep Policy Gradients*.
- [ ] Vinyals et al. 2019 (AlphaStar) / Berner et al. 2019 (OpenAI Five).


## 2. Policy parameterization: shared MLP over RAM, not a CNN over pixels

**Problem.** What does the actor-critic network look like?

**Options considered.**
- CNN over the 210×160×3 pixel observation (the classical Atari setup).
- MLP over the 128-byte RAM observation.
- Recurrent (LSTM/GRU) variants of either.
- Separate networks per agent slot vs. a single shared network.

**Decision.** A small **MLP** taking the 128-byte RAM vector, with a shared trunk and two heads (policy logits over 6 actions; scalar value). **One** network is shared across all 4 agent slots (parameter sharing).

**Why (to source).**
- The RAM observation is already a compact game-state summary — no need to re-learn vision from pixels. Much smaller network, much faster training.
- The 4 Warlords players are symmetric, so parameter sharing is the standard simplification in symmetric multi-agent RL — it cuts sample complexity ~4× and is invariant to the trivial "which player am I" relabeling.
- No recurrence: RAM already encodes most game state, and adding an LSTM significantly complicates the rollout buffer and the PPO update.

**Where it lives.** `network.py::ActorCritic`.

*Sources to add:*
- [ ] PettingZoo / ALE docs on RAM observations.
- [ ] Gupta et al. 2017, *Cooperative Multi-Agent Control Using Deep RL* — parameter sharing.
- [ ] Terry et al., PettingZoo paper.


## 3. Advantage estimation: GAE

**Problem.** PPO needs an advantage estimate $$\hat{A}_t$$. How do we compute it?

**Options considered.**
- Monte-Carlo returns minus $$V(s_t)$$ — unbiased but very high variance.
- One-step TD advantage $$r_t + \gamma V(s_{t+1}) - V(s_t)$$ — low variance, high bias.
- *n*-step returns — manual bias/variance tradeoff.
- **Generalized Advantage Estimation (GAE)** — exponentially-weighted average of all *n*-step returns, controlled by $$\lambda \in [0, 1]$$.

**Decision.** **GAE** with separate $$\gamma$$ (discount) and $$\lambda$$ (GAE bias/variance) hyperparameters, computed via the standard backwards recursion

$$\hat{A}_t = \delta_t + (\gamma\lambda)\,\hat{A}_{t+1},\qquad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t).$$

**Why (to source).**
- GAE is the standard advantage estimator paired with PPO — the original PPO paper itself uses it.
- The $$\lambda$$ knob is exactly what we need to tune later for delayed rewards (Section 7).

**Where it lives.** `gae.py::compute_gae`, called from `ppo_agent.py`.

*Sources to add:*
- [ ] Schulman et al. 2016, *High-Dimensional Continuous Control Using GAE* (arXiv:1506.02438).


## 4. Rollout buffer: timestep-major (T × N) layout

**Problem.** We're collecting on-policy rollouts from $$N$$ parallel envs for $$T$$ steps. How do we store them?

**Options considered.**
- Flat `(T·N,)` arrays.
- Env-major `(N, T)` arrays.
- Timestep-major `(T, N)` arrays.

**Decision.** Timestep-major `(T, N, ...)`.

**Why.**
- GAE is computed *backwards in time*; a `(T, N)` layout means the recursion is a contiguous vectorized sweep over the leading axis with $$N$$-wide batched updates per step.
- Episode-boundary masks (`dones`) align naturally as `(T, N)` and broadcast cleanly into the GAE recursion.
- Easy to flatten to `(T·N, ...)` for the PPO minibatch update afterwards.

**Where it lives.** `buffer.py::RolloutBuffer`.

*Sources to add (optional):*
- [ ] Any standard PPO implementation walkthrough (e.g., CleanRL, Stable-Baselines3 docs) for the buffer-layout convention.


## 5. Numerical stability of PPO → discovering HEPPO

**Problem.** A standard PPO implementation on Warlords showed the usual known PPO failure modes: value-loss scale dominating the policy loss, exploding early rewards skewing the running statistics, etc. This is a documented general PPO issue, not specific to Warlords.

**How HEPPO entered the picture.** While searching for principled ways to stabilize PPO's reward / value scale, we found **HEPPO** (Taha & Abdelhadi, arXiv:2501.12703). Its headline is an FPGA accelerator, but **Section II** of the paper is a set of *purely algorithmic* numerical-stability tricks that transfer cleanly to a software implementation:

1. **Dynamic reward standardization** using Welford's online mean/variance.
2. **Block standardization of values** (per-rollout), with explicit de-standardization before they re-enter GAE.
3. **Uniform *n*-bit quantization** of standardized values.

We adopted (1)-(3) as our PPO numerics layer. We did **not** implement the hardware contributions (the k-step lookahead pipelining trick is a clock-cycle re-association; it computes the same recursion in software).

**Why this is worth doing in software (to source).**
- Reward standardization addresses the well-known fact that PPO is sensitive to reward scale (Engstrom et al. 2020; Andrychowicz et al. 2021).
- Value standardization keeps the value-loss term on a comparable scale to the policy loss, which is otherwise a fragile hyperparameter (`vf_coef`).
- Quantization is the one that's *primarily* there for the hardware story, but the paper also reports it as a mild regularizer in software — we keep it as an ablatable knob (`--quant-bits`, `--no-quantization`).

**Where it lives.** `standardization.py` (all three primitives); wired into `ppo_agent.py` around the GAE call.

*Sources to add:*
- [ ] Taha & Abdelhadi 2025, *HEPPO: Hardware-Efficient PPO* (arXiv:2501.12703).
- [ ] Welford 1962 — online mean/variance algorithm.
- [ ] Engstrom et al. 2020, *Implementation Matters in Deep Policy Gradients*.
- [ ] Andrychowicz et al. 2021, *What Matters in On-Policy RL*.


In [ ]:
# Sanity-check Welford's online stats against the closed-form mean/std.
import numpy as np

rng = np.random.default_rng(0)
stream = rng.normal(loc=3.0, scale=2.0, size=10_000)

n, mean, M2 = 0, 0.0, 0.0
for x in stream:
    n += 1
    delta = x - mean
    mean += delta / n
    M2 += delta * (x - mean)

var = M2 / (n - 1)
print(f'Welford : mean={mean:.6f}  std={np.sqrt(var):.6f}')
print(f'NumPy   : mean={stream.mean():.6f}  std={stream.std(ddof=1):.6f}')

## 6. Advantage standardization (per-batch)

**Problem.** Even with standardized rewards and values, the *advantages* fed to the policy loss can drift in scale between rollouts.

**Decision.** Standardize advantages to mean 0 / std 1 *per minibatch* before the PPO clipped objective. Toggleable via `--no-adv-standardize`.

**Why (to source).**
- This is a near-universal PPO implementation detail; it's orthogonal to the HEPPO standardizations and exists purely as a policy-gradient numerics stabilizer.
- Worth keeping it ablatable so we can actually *show* its effect.

**Where it lives.** `ppo_agent.py::HEPPOAgent.update`.

*Sources:*
- [ ] Schulman et al. 2017 (PPO) — original mention.
- [ ] Huang et al. 2022, *The 37 Implementation Details of PPO*.


## 7. The reward problem in *Warlords*: delayed terminal ±1

**Problem.** `warlords_v3` gives reward 0 every step, then a single **+1 (win) / -1 (lose)** at episode end. Episodes are long (hundreds of steps). Standard PPO defaults (`gamma=0.99`) make this signal essentially invisible: a reward 500 steps in the past contributes $$0.99^{500} \approx 0.0066$$ to the value at $$t$$, which is in the noise floor of the value head.

**Options considered.**
1. **Reward shaping** — add dense surrogate signals (distance to fortress, bricks remaining, etc.).
2. **Curiosity / intrinsic motivation** (RND, ICM).
3. **Larger $$\gamma$$ + larger $$\lambda$$** to extend the effective credit-assignment horizon.
4. **Hindsight / return-conditioning** methods.

**Decision.** **Option 3.** Raise the defaults to `gamma=0.999, lambda=0.97`. Leave a *hook* for reward shaping (`PPOConfig.use_reward_shaping`) but do **not** enable it by default.

**Why (to source).**
- Reward shaping a *competitive win/lose* objective is risky: the agent may optimize the surrogate (e.g., "break bricks") instead of actually winning. Badly shaped rewards can change the optimal policy (Ng et al. 1999 describe the potential-based conditions under which shaping is policy-invariant).
- Curiosity is overkill: exploration isn't the bottleneck here, *credit assignment* is.
- Increasing $$\gamma$$ directly addresses the problem (it's what the discount-factor knob is for) and is mechanically simple. $$\lambda$$ trades more bias for less variance in GAE; raising it slightly helps when the reward signal is concentrated at the end.
- Concretely: $$0.999^{500} \approx 0.61$$ vs. $$0.99^{500} \approx 0.0066$$ — about a 90× larger effective signal at 500 steps back. See the plot below.

**Where it lives.** `ppo_agent.py::PPOConfig` defaults; `gae.py` consumes them.

*Sources to add:*
- [ ] Ng, Harada & Russell 1999, *Policy Invariance Under Reward Transformations*.
- [ ] Sutton & Barto, *Reinforcement Learning: An Introduction* — discounting / effective horizon.
- [ ] Pathak et al. 2017 (ICM) / Burda et al. 2018 (RND).
- [ ] Any empirical paper on PPO's sensitivity to $$\gamma$$ in sparse-reward settings.


In [ ]:
# How far back in time does a single terminal reward 'reach'?
import numpy as np
import matplotlib.pyplot as plt

T = 1000
t = np.arange(T)
for gamma in (0.99, 0.995, 0.999):
    plt.plot(t, gamma ** t, label=f'gamma={gamma}')
plt.axhline(0.01, color='gray', linestyle='--', label='1% of peak')
plt.xlabel('steps before terminal reward')
plt.ylabel('discounted contribution')
plt.yscale('log')
plt.legend()
plt.title('Effective credit-assignment horizon as a function of gamma')
plt.show()

## 8. From 'all 4 train together' to self-play

**Problem.** *Warlords* has 4 agent slots. The simplest setup — used in `train_warlords.py` — has one shared network playing all 4 slots, so every rollout produces 4× the data per env step. But under the real +1/-1 reward, this means the network is simultaneously being told "this trajectory won" and "this trajectory lost" for the *same* policy, which is exactly the zero-sum-against-itself degeneracy.

**Options considered.**
- **Naive parameter sharing** — one network, all 4 slots train (the baseline).
- **Independent policies** — 4 separate networks. Quadruples parameters and loses the symmetry simplification; opponents are still always co-evolving, which is unstable.
- **Self-play with a fixed opponent** — one trainee, 3 frozen copies. Risk of overfitting to that one snapshot.
- **Self-play with an opponent *pool*** — one trainee, 3 opponents drawn from a pool of past trainee snapshots.
- **League / PFSP** — full league with main agents, exploiters, etc. (AlphaStar-scale).

**Decision.** **Self-play with a snapshot pool** (`opponent_pool.py`, `train_warlords_selfplay.py`). One trainee slot per env; the other 3 are filled by frozen samples from a pool of past trainee checkpoints.

**Why (to source).**
- Self-play with past snapshots is the standard way to stabilize a multi-agent competitive learner — it avoids the chasing-a-moving-target pathology of pure co-evolution.
- Pooling past versions specifically prevents the policy from forgetting how to beat old strategies ("strategy cycling").
- A full league is overkill for a course-scale project; a snapshot pool is the minimum viable version of the same idea.

**Where it lives.** `opponent_pool.py`, `train_warlords_selfplay.py`.

*Sources to add:*
- [ ] Silver et al. 2017/2018 (AlphaGo Zero / AlphaZero).
- [ ] Vinyals et al. 2019, *Grandmaster level in StarCraft II* (AlphaStar) — league + PFSP.
- [ ] Berner et al. 2019, *Dota 2 with Large Scale Deep RL* (OpenAI Five) — past-version pool.
- [ ] Bansal et al. 2018, *Emergent Complexity via Multi-Agent Competition*.
- [ ] Heinrich & Silver 2016, *Deep RL from Self-Play in Imperfect-Information Games* (NFSP).


## 9. Opponent sampling: ELO-weighted, not uniform

**Problem.** Given a pool of past snapshots, *which* ones do we sample as the 3 opponents each rollout?

**Options considered.**
- **Always the latest snapshot.** Forgetting-prone; the trainee overfits to one opponent.
- **Uniform random over the pool.** Wastes rollouts on huge skill mismatches where the advantage signal is near-constant (blowouts in either direction).
- **ELO-weighted sampling** — probability concentrates on opponents close in rating to the trainee.
- **Prioritized Fictitious Self-Play (PFSP)** — weight by *loss probability* against the trainee, to actively seek out the trainee's weaknesses.

**Decision.** **ELO-weighted sampling** with a temperature parameter (`--elo-temperature`). This is a lightweight version of PFSP that requires only the rating, not per-opponent win-rate estimates.

**Eviction policy:** evict the *oldest* snapshot when the pool is full, **not** the weakest — keeping a spread of past "eras" prevents collapse onto only recent strong opponents.

**Why (to source).**
- The most informative games are those between roughly evenly matched opponents — the gradient signal in a 50/50 matchup is much larger than in a blowout.
- ELO is the cheapest sufficient statistic for skill.
- Keeping old snapshots specifically targets the strategy-cycling failure.

**Where it lives.** `opponent_pool.py`.

*Sources to add:*
- [ ] Vinyals et al. 2019 — PFSP description.
- [ ] Balduzzi et al. 2019, *Open-ended Learning in Symmetric Zero-Sum Games*.
- [ ] Elo, *The Rating of Chessplayers, Past and Present* (1978).


## 10. Progress metric: ELO, not mean return

**Problem.** How do we know the agent is *actually* getting better?

**The trap.** Under self-play, the opponent distribution keeps changing. Mean episode return mixes "the trainee got stronger" with "the opponents happened to be weaker this iteration." A rising return curve can come from either.

**Decision.** Track **ELO** of the current trainee against the pool as the primary progress metric, plus a rolling win rate. Raw return is logged but explicitly de-emphasized.

**Why.** ELO is constructed exactly to normalize for who you played — that's the whole reason chess uses it. Same problem, same solution.

**Where it lives.** `opponent_pool.py::EloTracker`, logged in `train_warlords_selfplay.py`.

*Sources to add:*
- [ ] Elo 1978.
- [ ] Silver et al. 2018 (AlphaZero) — ELO as the canonical self-play progress metric.


## 11. Do the HEPPO standardizations still apply under self-play?

**Yes, unchanged.** Worth stating explicitly because the two design layers interact:

- **Dynamic reward standardization** still runs on the ±1/0 stream. Under self-play, its running mean/std partially absorbs *opponent-pool difficulty drift* in addition to the trainee's own skill changes. That's the intended non-stationarity-handling behavior — but it also means the standardized reward stream is **not** a skill metric (use ELO, see §10).
- **Block value standardization + de-standardization** is purely a PPO numerics layer; orthogonal to self-play.
- **Advantage standardization** (§6) likewise: orthogonal, always on.

*Sources:* HEPPO paper (already listed in §5).


## 12. Ablation harness

**Problem.** We've made several decisions on top of vanilla PPO. Which of them actually matter on Warlords?

**Decision.** `run_ablation.py` reproduces the 5 configurations from HEPPO's Table III back-to-back, plus the orthogonal toggles `--no-quantization`, `--no-adv-standardize`, `--quant-bits N`, `--use-reward-shaping`. Compare on mean episode return for the dense-baseline script and on **ELO trajectory** for the self-play script.

**Why.** Decisions without ablations are just opinions. Even when the expected ordering is known from the source paper, re-running it on Warlords is what tells us the decision transfers to *this* environment.

**Where it lives.** `run_ablation.py`.

*Sources:* HEPPO Table III (§5).


## 13. Testing without Atari ROMs

**Problem.** The development sandbox can't reach the AutoROM CDN, so the Atari ROMs aren't available during CI/dev.

**Decision.** Test against a **mock of PettingZoo's parallel API** that reproduces the exact agent-death / episode-end bookkeeping (including the asymmetric case where the trainee dies before the other 3 finish). All other components are tested on synthetic data shaped exactly like a Warlords RAM rollout (128-byte obs, 6 actions):

- Welford's running stats vs. closed-form mean/std.
- Block standardization round-trip.
- Quantization error scaling with bit-width.
- GAE recursion vs. independent reference implementation (with done-masking).
- ELO updates: zero-sum, upsets weighted more than expected wins.
- Pool eviction policy.
- End-to-end PPO update on a toy bandit-style task (action preference goes from ~17% to 100% in ~20 iterations) — confirms the wiring produces a real learning signal independent of Warlords.

**Where it lives.** Test scripts (to be added next to each module).

*Sources:* PettingZoo parallel-API docs (for the mock contract).


## Summary table

| # | Decision | Alternative rejected | Primary reason | File |
|---|---|---|---|---|
| 1 | PPO-Clip | DQN, A2C, TRPO, SAC, MuZero | on-policy + stochastic + simple + self-play track record | `ppo_agent.py` |
| 2 | Shared MLP over RAM | CNN over pixels; per-agent nets | RAM is compact; agents are symmetric | `network.py` |
| 3 | GAE | MC / TD / n-step | bias/variance knob via $$\lambda$$ | `gae.py` |
| 4 | (T, N) buffer layout | flat / env-major | matches backward GAE sweep | `buffer.py` |
| 5 | HEPPO standardizations | naive value-loss scaling | known PPO reward-scale fragility | `standardization.py` |
| 6 | Advantage standardization | none | per-batch policy-gradient stability | `ppo_agent.py` |
| 7 | $$\gamma=0.999, \lambda=0.97$$ | reward shaping; curiosity | preserves the actual objective | `ppo_agent.py` |
| 8 | Self-play with snapshot pool | all 4 slots train; co-evolution | breaks moving-target instability | `opponent_pool.py` |
| 9 | ELO-weighted sampling | uniform; latest-only; full PFSP | informative matchups; cheap | `opponent_pool.py` |
| 10 | ELO as progress metric | mean return | normalizes for opponent difficulty | `train_warlords_selfplay.py` |
| 11 | Keep HEPPO under self-play | turn it off | orthogonal numerics layer | `ppo_agent.py` |
| 12 | Explicit ablation script | trust the paper | local validity on Warlords | `run_ablation.py` |
| 13 | Mocked PettingZoo tests | no tests | sandbox has no ROM access | (tests) |

---

### How to use this template

1. Fill in the *Sources to add* bullets under each decision as you find them.
2. If a decision changes during the project, **don't delete the old reasoning** — add a dated note ("Revised on YYYY-MM-DD because …"). The audit trail is the point of this document.
3. Keep the structure: Problem → Options → Decision → Why → Where → Sources. Reviewers (and future-you) should be able to scan any section in 30 seconds.